# Lab 2 — Build a Working Collaborative-Filtering Recommender
**ACM Workshop · Recommender Systems · Day 1 · 11:45–12:45 (60 min)**

Just taught: *similar users have similar tastes* →
$$\hat r_{ui}=\bar r_u+\frac{\sum_{v\in N(u)} \mathrm{sim}(u,v)\,(r_{vi}-\bar r_v)}{\sum_{v\in N(u)}|\mathrm{sim}(u,v)|}$$

**You will build** (with NumPy/scikit-learn only — no recommender library yet):
1. similarity from a toy example → then from real data
2. **user-based CF**: neighbours → predictions → Top-N for user 10
3. **item-based CF**: similar movies → predictions
4. an honest **evaluation** (RMSE vs baselines) and an experiment on **k**
5. a **cold-start** demo — pretend to be a brand-new user

> Pipeline position: Raw data → EDA → **similarity → collaborative filtering** → MF → prediction → top-N → evaluation

## 0. Setup & data (the split from Lab 1)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 20, "display.width", 140, "display.max_colwidth", 60)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def find_proc():
    for base in ["data", "../data", "../../data", "../../../data"]:
        if (Path(base) / "processed" / "train.csv").exists(): return Path(base) / "processed"
    raise FileNotFoundError("Run Lab 1 first (it creates data/processed/train.csv).")
PROC = find_proc(); FIG = PROC.parent.parent / "figures"; FIG.mkdir(exist_ok=True)

train  = pd.read_csv(PROC / "train.csv")
test   = pd.read_csv(PROC / "test.csv")
movies = pd.read_csv(PROC / "movies_clean.csv").set_index("movieId")
title  = movies.title
print(len(train), "train ratings |", len(test), "test ratings")

## 1. Similarity on a toy example (the slide example)
Three users, three movies. Who is similar to A?

In [ ]:
toy = pd.DataFrame({"LOTR": [5, 5, 1], "Matrix": [5, 4, 1], "Titanic": [1, 1, 5]}, index=list("ABC"))
print(toy, "\n")
sim = pd.DataFrame(cosine_similarity(toy), index=toy.index, columns=toy.index).round(3)
print("cosine similarity:\n", sim)

By hand for A and B: $\cos = \dfrac{5\cdot5+5\cdot4+1\cdot1}{\sqrt{51}\,\sqrt{42}} = \dfrac{46}{46.28}\approx 0.994$.
**But** look at A vs C: cosine is still ~0.40 although they disagree on everything. Raw cosine on ratings is high whenever all numbers are positive.
The fix: **mean-centre** each user's ratings first (subtract the user's average) — then "above average" is positive and "below average" is negative.

In [ ]:
centred = toy.sub(toy.mean(axis=1), axis=0)
print(centred.round(2), "\n")
print("cosine on centred ratings:\n", pd.DataFrame(cosine_similarity(centred), index=toy.index, columns=toy.index).round(3))

→ Now A and B are almost identical (+0.97) and A and C are opposites (−1.0). This is (nearly) the **Pearson correlation**.

## 2. User–item matrix from the training data
Same as Lab 1 but in plain NumPy for speed. `NaN` = unknown. **Only training ratings** go in — the test ratings stay hidden.

In [ ]:
all_items = np.sort(pd.concat([train.movieId, test.movieId]).unique())
users     = np.sort(train.userId.unique())
u_idx = {u: k for k, u in enumerate(users)}; i_idx = {i: k for k, i in enumerate(all_items)}

R = np.full((len(users), len(all_items)), np.nan)
R[train.userId.map(u_idx), train.movieId.map(i_idx)] = train.rating.values

M    = ~np.isnan(R)                      # mask: True where a rating is known
mu_u = np.nanmean(R, axis=1)             # each user's mean rating
Rc   = np.where(M, R - mu_u[:, None], 0.0)   # mean-centred, unknown -> 0
n_item = M.sum(axis=0)                   # popularity (training ratings per movie)
print("R:", R.shape, "| known cells:", M.sum(), f"| sparsity {1 - M.mean():.2%}")

## 3. User-based CF
### 3.1 Similarity between users
Cosine similarity of the **mean-centred** rows. Unknown cells are 0, i.e. "no opinion", so they do not contribute to the dot product.

In [ ]:
norm_u = np.linalg.norm(Rc, axis=1)
Mf = M.astype(float)
overlap_uu = Mf @ Mf.T                                     # number of movies each pair of users has in common

def build_S_uu(shrink=None):
    S = (Rc @ Rc.T) / (np.outer(norm_u, norm_u) + 1e-9)    # cosine of mean-centred rows
    if shrink: S = S * np.minimum(1, overlap_uu / shrink)   # significance weighting (optional)
    np.fill_diagonal(S, 0)                                  # a user is not their own neighbour
    return S
S_uu = build_S_uu()

def user_sims(vec):
    # similarity of ANY (possibly partial / brand-new) rating vector to every user in the matrix
    m = ~np.isnan(vec); v = np.where(m, vec - np.nanmean(vec), 0.0)
    return Rc @ v / (norm_u * np.linalg.norm(v) + 1e-9)

TARGET = 10
s = S_uu[u_idx[TARGET]]
nn = np.argsort(-s)[:10]
pd.DataFrame({"neighbour userId": users[nn], "similarity": s[nn].round(3),
              "# movies in common": overlap_uu[u_idx[TARGET], nn].astype(int), "# ratings": M[nn].sum(1)})

**Look carefully**: some neighbours share only a handful of movies with user 10 → the similarity is *unreliable*.
A common fix is **significance weighting** — shrink similarity when the overlap is small: $\mathrm{sim}' = \mathrm{sim}\cdot\min(1, n_{uv}/50)$. We will try it in an exercise.

### 3.2 What does user 10 like? (so we can judge the recommendations)

In [ ]:
def history(u, n=8):
    row = R[u_idx[u]]; ids = all_items[~np.isnan(row)]
    h = pd.DataFrame({"title": title.reindex(ids).values, "rating": row[~np.isnan(row)]}, index=ids)
    return h.sort_values("rating", ascending=False).head(n)
print(f"User {TARGET} has {M[u_idx[TARGET]].sum()} training ratings, mean {mu_u[u_idx[TARGET]]:.2f}. Favourites:")
history(TARGET)

### 3.3 Predict + recommend
$$\hat r_{ui}=\bar r_u+\frac{\sum_{v\in N_k(u,i)} s_{uv}\,(r_{vi}-\bar r_v)}{\sum_{v\in N_k(u,i)} |s_{uv}|}$$
$N_k(u,i)$ = the **k most similar users *who rated movie i*** (only positively-similar ones are used).

In [ ]:
raters = [np.where(M[:, j])[0] for j in range(M.shape[1])]       # who rated each movie (training data)

def predict_ui(s, j, k=20):
    # s: similarity of the target user to all users; j: movie column.  Returns (prediction offset, support)
    r = raters[j]
    if len(r) == 0: return np.nan, 0
    sr = s[r]
    if len(r) > k:
        top = np.argpartition(-sr, k - 1)[:k]; r, sr = r[top], sr[top]
    pos = sr > 0
    if not pos.any(): return np.nan, 0
    r, sr = r[pos], sr[pos]
    return (sr @ Rc[r, j]) / np.abs(sr).sum(), int(pos.sum())     # weighted average of centred ratings

def recommend_from_sims(s, base, seen, k=20, n=5, min_pop=20, min_support=3):
    # s: similarities to all users, base: the user's mean rating, seen: mask of movies already rated
    out = []
    for j in np.where(~seen & (n_item >= min_pop))[0]:
        off, sup = predict_ui(s, j, k)
        if sup >= min_support: out.append((j, base + off, sup))
    out = sorted(out, key=lambda t: -t[1])[:n]
    return pd.DataFrame({"title": [title[all_items[j]] for j, _, _ in out], "pred_rating": [round(min(5, p), 2) for _, p, _ in out],
                         "# neighbours used": [c for _, _, c in out], "# ratings overall": [n_item[j] for j, _, _ in out]},
                        index=[all_items[j] for j, _, _ in out])

def recommend_user_cf(u, **kw):
    i = u_idx[u]; return recommend_from_sims(S_uu[i], mu_u[i], M[i], **kw)

print(f"Top-5 for user {TARGET}  (k=20 neighbours)")
recommend_user_cf(TARGET, k=20)

**What are `min_pop` and `min_support` doing?** Without them, the highest predictions are movies that *one* neighbour rated 5★. They are a guard against the unreliable-average problem from Lab 1 (§5).
Try `min_pop=0, min_support=1` to see the effect → it is dramatic.

In [ ]:
print("WITHOUT the support guards (min_pop=0, min_support=1):")
recommend_user_cf(TARGET, k=20, min_pop=0, min_support=1)

### 3.4 Discussion experiment: how many neighbours?

In [ ]:
lists = {k: recommend_user_cf(TARGET, k=k, n=5, min_support=1 if k == 2 else 3) for k in (2, 20, 50)}
for k, df in lists.items():
    print(f"\n--- k = {k} ---"); print(df[["title", "pred_rating", "# neighbours used"]].to_string())

**Discuss in pairs (3 min):**
- With **k = 2** the list depends on just two people → noisy; the predictions are inflated (4.7+) because two strangers' opinions are not averaged with anything.
- **k = 20 and k = 50 give the *identical* list. Why?** Look at the last column: each candidate was rated by only 3–10 of the positively-similar users. When fewer than *k* neighbours have rated a movie, increasing *k* changes nothing — **sparsity, not *k*, limits the neighbourhood.**
- So *k* trades **variance** (small k: noisy) against **bias** (large k: less personal) — but only until the data runs out. We measure the effect on error in §5.

## 4. Item-based CF
> "Which movies are similar to the movies I already liked?"

Item–item similarity: cosine of the **columns** of the (user-mean-centred) matrix — the *adjusted cosine*.
We restrict to movies with ≥ 10 training ratings (similarity from 1–2 ratings is meaningless, and the full matrix would not fit in memory).

In [ ]:
MIN_ITEM = 10
keep = np.where(n_item >= MIN_ITEM)[0]
print(f"{len(keep)} movies have >= {MIN_ITEM} training ratings (of {len(all_items)})")
Xi = Rc[:, keep]
nrm = np.linalg.norm(Xi, axis=0) + 1e-9
S_item = (Xi.T @ Xi) / np.outer(nrm, nrm)
np.fill_diagonal(S_item, 0)
kept_ids = all_items[keep]; k_idx = {i: j for j, i in enumerate(kept_ids)}

def similar_movies(movie_id, n=8):
    s = pd.Series(S_item[k_idx[movie_id]], index=kept_ids).nlargest(n)
    return pd.DataFrame({"title": title.reindex(s.index).values, "similarity": s.round(3).values,
                         "# ratings": n_item[[i_idx[i] for i in s.index]]}, index=s.index)

print("Movies most similar to:", title[1]); display(similar_movies(1))
print("Movies most similar to:", title[2571]); display(similar_movies(2571))

No genre information was used — *Toy Story's* neighbours are other animated/family films purely because **the same people rated them similarly**. That is the "collaborative" part.

In [ ]:
# for every user: which (kept) movies they rated and their centred ratings
rated_k = [np.where(M[u, keep])[0] for u in range(len(users))]
rated_c = [Rc[u, keep][rated_k[u]] for u in range(len(users))]

def predict_item_based(u, j, k=20):
    # j: index in the `keep` space. Neighbours = the k movies MOST SIMILAR TO j among those user u has rated.
    idx = rated_k[u]
    if len(idx) == 0: return np.nan, 0
    sj = S_item[j, idx]; c = rated_c[u]
    if len(idx) > k:
        top = np.argpartition(-sj, k - 1)[:k]; sj, c = sj[top], c[top]
    pos = sj > 0
    if not pos.any(): return np.nan, 0
    return (sj[pos] @ c[pos]) / sj[pos].sum(), int(pos.sum())

def recommend_item_cf(u, k=20, n=5, min_support=3, min_pop=30):
    ui = u_idx[u]; out = []
    for j in np.where(~M[ui, keep] & (n_item[keep] >= min_pop))[0]:
        off, sup = predict_item_based(ui, j, k)
        if sup >= min_support: out.append((j, mu_u[ui] + off, sup))
    out = sorted(out, key=lambda t: -t[1])[:n]
    ids = [kept_ids[j] for j, _, _ in out]
    return pd.DataFrame({"title": title.reindex(ids).values, "pred_rating": [round(min(5, p), 2) for _, p, _ in out],
                         "# similar movies used": [c for _, _, c in out]}, index=ids)
print(f"Item-based Top-5 for user {TARGET}"); recommend_item_cf(TARGET)

Compare with §3.3. Item-based lists tend to be *"more of what you already like"* (safer, more stable), user-based lists can be more surprising.

## 5. Evaluation — do the numbers agree with our intuition?
Predict every **test** rating, compare with the truth. Always include **baselines**: if you cannot beat "predict the average", you have not built a recommender.

In [ ]:
test_u = test.userId.map(u_idx).values; test_i = test.movieId.map(i_idx).values; y = test.rating.values
def evaluate(pred, name):
    p = np.where(np.isnan(pred), mu_u[test_u], pred).clip(0.5, 5)      # fall back to the user's mean when CF has no answer
    return {"model": name, "RMSE": np.sqrt(np.mean((y - p) ** 2)), "MAE": np.mean(np.abs(y - p)), "coverage": 1 - np.isnan(pred).mean()}

def pred_user_cf(k, S=None):
    S = S_uu if S is None else S
    return np.array([mu_u[u] + predict_ui(S[u], j, k)[0] for u, j in zip(test_u, test_i)])

kept_pos = {j: p for p, j in enumerate(keep)}
def pred_item_cf(k):
    out = np.full(len(test_u), np.nan)
    for n_, (u, j) in enumerate(zip(test_u, test_i)):
        if j in kept_pos: out[n_] = mu_u[u] + predict_item_based(u, kept_pos[j], k)[0]
    return out

g_mean = train.rating.mean()
item_mean = np.where(n_item > 0, np.nansum(R, 0) / np.maximum(n_item, 1), g_mean)
rows = [evaluate(np.full(len(y), g_mean), "baseline: global mean"),
        evaluate(mu_u[test_u], "baseline: user mean"),
        evaluate(item_mean[test_i], "baseline: item mean"),
        evaluate(pred_user_cf(20), "user-based CF (k=20)"),
        evaluate(pred_user_cf(50), "user-based CF (k=50)"),
        evaluate(pred_item_cf(20), "item-based CF (k=20)")]
res = pd.DataFrame(rows).set_index("model"); res.round(4)

**Read it honestly.** Similarity-based CF beats every baseline, but the gap to the *user-mean* baseline (~0.06 RMSE) is smaller than people expect. *Coverage* differs too: item-based CF only answers for movies with ≥ 10 ratings (≈75 % of test pairs; the rest fall back to the user mean), so compare RMSEs with care.
Part of the signal is just "this user is generous / this movie is good" (**biases**) — matrix factorization (next session) models those explicitly.

In [ ]:
ks = [1, 2, 5, 10, 20, 50, 100, 200]
curve = pd.DataFrame([evaluate(pred_user_cf(k), f"k={k}") | {"k": k} for k in ks]).set_index("k")
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(curve.index, curve.RMSE, "o-", color="#2b6cb0"); ax.set_xscale("log")
ax.axhline(res.loc["baseline: user mean", "RMSE"], ls="--", color="gray", label="user-mean baseline"); ax.legend()
ax.set(xlabel="k (neighbours)", ylabel="test RMSE", title="User-based CF: effect of k")
plt.tight_layout(); plt.savefig(FIG / "lab2_k_curve.png"); plt.show()
curve[["RMSE", "MAE", "coverage"]].round(4)

**Interpretation.** With *k = 1* we trust a single stranger → worse than predicting the global mean. Error drops fast up to *k ≈ 10–20*, then **flattens** (diminishing returns: extra neighbours have tiny similarity weights).
Choosing *k* is **hyper-parameter tuning**: do it on a *validation* set, never on the test set (here we show the test curve for teaching only).

## 6. Cold start — pretend to be a new user
A brand-new user has *no* row in the matrix. We ask for 3–5 favourite movies and run CF on that partial vector.

In [ ]:
def find(q): return movies[movies.title.str.contains(q, regex=False)].index[0]
my_ratings = {find("Matrix, The (1999)"): 5, find("Star Wars: Episode IV"): 5, find("Alien (1979)"): 4.5, find("Toy Story (1995)"): 2}
vec = np.full(R.shape[1], np.nan)
for mid, r_ in my_ratings.items(): vec[i_idx[mid]] = r_
print("I am a new user. My ratings:", {title[m]: r for m, r in my_ratings.items()})

new_s = user_sims(vec)
print("\nRecommendations for the new user:")
recommend_from_sims(new_s, np.nanmean(vec), ~np.isnan(vec), k=30, n=8, min_pop=50)

With only four ratings we already get a (rough) personalised list. With **zero** ratings CF has nothing to work with — the standard fallback is **popularity**:

In [ ]:
# Popularity fallback: Bayesian-damped mean ("IMDb-style" weighted rating)
C, m_ = g_mean, 30
agg = train.groupby("movieId").rating.agg(["mean", "count"])
agg["score"] = (agg["count"] * agg["mean"] + m_ * C) / (agg["count"] + m_)
agg.nlargest(8, "score").assign(title=lambda d: title.reindex(d.index).values)[["title", "count", "mean", "score"]].round(2)

## 7. Exercises (pick one, 10 min)
**E1 — Significance weighting.** `build_S_uu(shrink)` multiplies each similarity by `min(1, overlap/shrink)`. Predict first: will RMSE at k=20 improve? Then run it. *(Ideas from papers must be validated on your data!)*
**E2 — Explain a recommendation.** For user 10's #1 recommendation, list which neighbours contributed and what they rated.
**E3 — Another user.** Run the Top-5 for a user with *few* ratings and one with *thousands* (Lab 1: users 414, 599). What differs?

In [ ]:
# E2 — explainability: who drove the #1 recommendation?
# TODO: write your code here


In [ ]:
# E1 — significance weighting: shrink similarities that rest on few co-rated movies
# TODO: write your code here


## 8. Limitations we just *felt* (→ next theory session)
| Observed | Name | What fixes it |
|---|---|---|
| Neighbours share few movies | **sparsity** | compress users/items into dense vectors |
| §3.3 junk without support filters | **unreliable similarity / popularity bias** | regularisation |
| New user needed hand-built vector | **cold start** | content features, hybrids |
| `user_cf_matrix` loops over every user; item-sim matrix is $n^2$ | **scalability** | learn a *small* number of factors |
| Predictions only as good as "bias + neighbours" | **bias is not modelled** | $b_u$, $b_i$ in matrix factorization |

## ✅ Lab 2 deliverable — checklist
- [x] user–user and item–item similarity · [x] Top-5 for user 10 (user-based and item-based) · [x] k experiment · [x] RMSE vs baselines · [x] new-user demo

**Next (13:30): Matrix Factorization — SVD / ALS.** *"What if we discover hidden factors that explain preferences instead of comparing users one by one?"*